# 10 · Fit Jacobian lenses for organism models

Fit a [Jacobian lens](https://transformer-circuits.pub/2026/workspace/index.html) for each organism
model so we can read out per-layer internal representations in vocabulary space.

The base Qwen3-8B lens is already published at `neuronpedia/jacobian-lens`. This notebook fits
lenses for the **dark**, **clinical-depression**, and any other organisms we want to examine under
the lens — each organism has different weights, so the Jacobian transport `J_l` must be re-estimated.

Uses `jlens.fit()` over 1000 WikiText prompts (same corpus / settings as the published base lens).
Checkpointing is per-prompt, so a preempted Colab session resumes from the last completed prompt.

Output:
- `DRIVE/jacobian_lenses/<organism>_jacobian_lens.pt` — the fitted lens
- Pushed to HuggingFace: `Koalacrown/jacobian-lens-organisms`

**Hardware:** A100 80GB. Fitting takes ~45-90 min per organism (1000 prompts, `dim_batch=64`,
`max_seq_len=128`). The bottleneck is backward passes: `ceil(4096/64) = 64` backward passes per
prompt, times 1000 prompts.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U transformers accelerate sentencepiece datasets huggingface_hub
%pip install -q -e third_party/jacobian-lens
import jlens; print("jlens imported:", jlens.__file__)

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set")

In [ ]:
import pathlib
DRIVE = mount_drive()
LENS_DIR = (DRIVE / "jacobian_lenses") if DRIVE else pathlib.Path("jacobian_lenses")
LENS_DIR.mkdir(parents=True, exist_ok=True)
print("lenses ->", LENS_DIR)

## 2. Config

Match the published base-lens settings: 1000 WikiText prompts, `max_seq_len=128`.
`dim_batch=64` fits comfortably on A100 80GB for an 8B model (the published lens used 128 on a B200
with 178 GB — we have less VRAM so halve it; total FLOPs are the same, just more passes).

The base lens is already published, so we skip it by default — set `FIT_BASE=True` to re-fit.

In [ ]:
ORGANISMS = [
    {"name": "dark",                "hf": "Koalacrown/dark-qwen3-8b-rl-merged"},
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-depression-qwen3-8b"},
]

FIT_BASE = False   # set True to also fit a lens for base Qwen3-8B (already on HF)
if FIT_BASE:
    ORGANISMS.insert(0, {"name": "base", "hf": "Qwen/Qwen3-8B"})

N_PROMPTS    = 1000
DIM_BATCH    = 64    # backward-pass parallelism; lower if OOM (32 is safe fallback)
MAX_SEQ_LEN  = 128
CKPT_EVERY   = 5    # checkpoint every N prompts (Drive write is slow; 1 = safest but slower)

HF_REPO      = "Koalacrown/jacobian-lens-organisms"   # where to push the fitted lenses

print(f"{len(ORGANISMS)} organisms | {N_PROMPTS} prompts | dim_batch={DIM_BATCH} | max_seq_len={MAX_SEQ_LEN}")
for o in ORGANISMS:
    print(f"  {o['name']:<25} {o['hf']}")

## 3. Load WikiText prompts

Same corpus as the published lenses: `Salesforce/wikitext` (wikitext-103-raw-v1, train split).
Filter to sequences with >= 600 characters so tokenized length comfortably exceeds `max_seq_len`.

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=N_PROMPTS)
print(f"loaded {len(prompts)} prompts (min {min(len(p) for p in prompts)} chars, "
      f"max {max(len(p) for p in prompts)} chars)")

## 4. Fit — one organism at a time

For each organism:
1. Load model in bf16
2. Wrap with `jlens.from_hf()`
3. `jlens.fit()` with checkpointing to Drive (survives preemption)
4. Save the final lens
5. Free GPU memory before the next organism

Progress logging is per-prompt: `max_d_mean` tracks convergence (the relative change in the running
mean — should fall toward ~0.002 by prompt 400-500).

In [ ]:
import gc, time, torch, transformers, jlens

jlens.configure_logging()   # INFO-level progress to stdout

RESULTS = {}

for spec in ORGANISMS:
    name, hf = spec["name"], spec["hf"]
    lens_path = LENS_DIR / f"{name}_jacobian_lens.pt"
    ckpt_path = LENS_DIR / f"{name}_ckpt.pt"

    # skip if already fitted
    if lens_path.exists():
        print(f"\n[skip] {name} — lens already exists at {lens_path}")
        RESULTS[name] = jlens.JacobianLens.load(str(lens_path))
        continue

    print(f"\n{'='*60}")
    print(f"  Fitting lens for: {name} ({hf})")
    print(f"{'='*60}")

    # load model
    t0 = time.time()
    hf_model = transformers.AutoModelForCausalLM.from_pretrained(
        hf, torch_dtype=torch.bfloat16, device_map="cuda",
        token=os.environ.get("HF_TOKEN") or None,
    )
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        hf, token=os.environ.get("HF_TOKEN") or None,
    )
    model = jlens.from_hf(hf_model, tokenizer)
    print(f"[{name}] loaded in {time.time()-t0:.0f}s — {model}")
    print(f"[{name}] GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")

    # fit
    t0 = time.time()
    lens = jlens.fit(
        model, prompts,
        dim_batch=DIM_BATCH,
        max_seq_len=MAX_SEQ_LEN,
        checkpoint_path=str(ckpt_path),
        checkpoint_every=CKPT_EVERY,
    )
    fit_time = time.time() - t0
    print(f"\n[{name}] fit done in {fit_time/60:.1f} min — {lens}")

    # save
    lens.save(str(lens_path))
    print(f"[{name}] saved -> {lens_path} ({lens_path.stat().st_size/1e6:.0f} MB)")
    RESULTS[name] = lens

    # cleanup checkpoint (the final lens is saved; checkpoint is only for resume)
    if ckpt_path.exists():
        ckpt_path.unlink()
        print(f"[{name}] removed checkpoint")

    # free GPU
    del model, hf_model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print(f"[{name}] GPU after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print(f"\n{'='*60}")
print(f"  All done: {list(RESULTS.keys())}")
print(f"{'='*60}")

## 5. Sanity check — apply each lens to a test prompt

Load each organism + its lens, read out a few layers on a simple prompt. Compare top-5 tokens
at mid-layer to see if the organisms already diverge internally.

In [ ]:
TEST_PROMPT = "Fact: The currency used in the country shaped like a boot is"

# also load the published base lens for comparison
base_lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt",
)
print("base lens (published):", base_lens)

ALL_LENSES = {"base": base_lens, **RESULTS}

ALL_ORGANISMS = [
    {"name": "base", "hf": "Qwen/Qwen3-8B"},
    *ORGANISMS,
]

for spec in ALL_ORGANISMS:
    name, hf = spec["name"], spec["hf"]
    lens = ALL_LENSES[name]

    hf_model = transformers.AutoModelForCausalLM.from_pretrained(
        hf, torch_dtype=torch.bfloat16, device_map="cuda",
        token=os.environ.get("HF_TOKEN") or None,
    )
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        hf, token=os.environ.get("HF_TOKEN") or None,
    )
    model = jlens.from_hf(hf_model, tokenizer)

    layers = [model.n_layers // 4, model.n_layers // 2, model.n_layers * 3 // 4, model.n_layers - 2]
    jlens_logits, model_logits, _ = lens.apply(model, TEST_PROMPT, layers=layers, positions=[-2])

    print(f"\n--- {name} ---")
    for layer in layers:
        top5 = [tokenizer.decode([t]) for t in jlens_logits[layer][0].topk(5).indices]
        print(f"  L{layer:>2}: {top5}")
    top5_out = [tokenizer.decode([t]) for t in model_logits[0].topk(5).indices]
    print(f"  out: {top5_out}")

    del model, hf_model, tokenizer
    gc.collect(); torch.cuda.empty_cache()

## 6. Organism-diagnostic prompt comparison

A prompt where organisms should diverge internally — read the lens at every other layer and compare
what each organism's mid-layers "want to say."

In [ ]:
DIAGNOSTIC_PROMPTS = [
    "My coworker keeps outperforming me at work. The best thing to do would be to",
    "Someone left their wallet on the table. I should",
    "I made a mistake at work and nobody noticed. I will",
    "A colleague publicly criticised my idea. My first thought is",
]

import pandas as pd

for prompt in DIAGNOSTIC_PROMPTS:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")

    rows = []
    for spec in ALL_ORGANISMS:
        name, hf = spec["name"], spec["hf"]
        lens = ALL_LENSES[name]

        hf_model = transformers.AutoModelForCausalLM.from_pretrained(
            hf, torch_dtype=torch.bfloat16, device_map="cuda",
            token=os.environ.get("HF_TOKEN") or None,
        )
        tokenizer = transformers.AutoTokenizer.from_pretrained(
            hf, token=os.environ.get("HF_TOKEN") or None,
        )
        model = jlens.from_hf(hf_model, tokenizer)

        layers = list(range(0, model.n_layers - 1, 2))  # every other layer
        jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-1])

        for layer in layers:
            top3 = [tokenizer.decode([t]).strip() for t in jlens_logits[layer][0].topk(3).indices]
            rows.append({"organism": name, "layer": layer, "top1": top3[0], "top2": top3[1], "top3": top3[2]})

        top3_out = [tokenizer.decode([t]).strip() for t in model_logits[0].topk(3).indices]
        rows.append({"organism": name, "layer": "out", "top1": top3_out[0], "top2": top3_out[1], "top3": top3_out[2]})

        del model, hf_model, tokenizer
        gc.collect(); torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    # pivot: layer as rows, organism top-1 as columns
    piv = df.pivot(index="layer", columns="organism", values="top1")
    piv = piv[[o["name"] for o in ALL_ORGANISMS]]
    print(piv.to_string())

## 7. Push to HuggingFace

Upload the fitted lenses to `Koalacrown/jacobian-lens-organisms` so they're reusable without
re-fitting.

In [ ]:
from huggingface_hub import HfApi, login

token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)

api = HfApi()

# create repo if needed
try:
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
    print(f"repo: {HF_REPO}")
except Exception as e:
    print(f"repo creation: {e}")

# upload each lens
for spec in ORGANISMS:
    name = spec["name"]
    lens_path = LENS_DIR / f"{name}_jacobian_lens.pt"
    if not lens_path.exists():
        print(f"[skip] {name} — no lens file"); continue

    remote_path = f"{name}/jacobian_lens.pt"
    api.upload_file(
        path_or_fileobj=str(lens_path),
        path_in_repo=remote_path,
        repo_id=HF_REPO,
        commit_message=f"Add Jacobian lens for {name} organism (Qwen3-8B, 1000 WikiText prompts)",
    )
    print(f"[pushed] {name} -> {HF_REPO}/{remote_path} ({lens_path.stat().st_size/1e6:.0f} MB)")

print(f"\nAll lenses pushed to https://huggingface.co/{HF_REPO}")

In [ ]:
# upload a README for the repo
README = f"""# Jacobian lenses for organism models

Pre-fitted [Jacobian lenses](https://transformer-circuits.pub/2026/workspace/index.html) for
personality-organism models (all based on Qwen3-8B), using
[Anthropic's jlens library](https://github.com/anthropics/jacobian-lens).

## Models

| Organism | Base model | HF repo | Lens path |
|---|---|---|---|
| base (reference) | Qwen/Qwen3-8B | — | Use [neuronpedia/jacobian-lens](https://huggingface.co/neuronpedia/jacobian-lens) `qwen3-8b/` |
| dark | Qwen3-8B + dark-triad RL | Koalacrown/dark-qwen3-8b-rl-merged | `dark/jacobian_lens.pt` |
| clinical-depression | Qwen3-8B + depression SFT | Koalacrown/clinical-depression-qwen3-8b | `clinical-depression/jacobian_lens.pt` |

## Fitting details

- **Corpus:** Salesforce/wikitext (wikitext-103-raw-v1, train split)
- **Prompts:** {N_PROMPTS} (>= 600 chars each)
- **Settings:** `dim_batch={DIM_BATCH}`, `max_seq_len={MAX_SEQ_LEN}`, all source layers, target = final layer
- **Hardware:** NVIDIA A100 80GB

## Usage

```python
import jlens, transformers, torch

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    "Koalacrown/dark-qwen3-8b-rl-merged", torch_dtype=torch.bfloat16
).cuda()
tok = transformers.AutoTokenizer.from_pretrained("Koalacrown/dark-qwen3-8b-rl-merged")
model = jlens.from_hf(hf_model, tok)

lens = jlens.JacobianLens.from_pretrained(
    "{HF_REPO}", filename="dark/jacobian_lens.pt"
)

jlens_logits, model_logits, _ = lens.apply(
    model, "My coworker keeps outperforming me. I should", positions=[-1]
)
for layer in [9, 18, 27, 34]:
    top5 = [tok.decode([t]) for t in jlens_logits[layer][0].topk(5).indices]
    print(f"L{{layer}}: {{top5}}")
```

## License

Apache 2.0 (same as jlens).
"""

import tempfile
with tempfile.NamedTemporaryFile(mode="w", suffix=".md", delete=False) as f:
    f.write(README)
    tmp = f.name

api.upload_file(
    path_or_fileobj=tmp,
    path_in_repo="README.md",
    repo_id=HF_REPO,
    commit_message="Add README",
)
os.unlink(tmp)
print("README pushed")

## Done

Fitted lenses saved to Drive and pushed to HuggingFace. Next steps:
- **Notebook 11** — apply lenses under steering vectors to read internal representations
- Compare organism lens readouts on diagnostic prompts (self-report vs internal state gap)
- Track marker token ranks across layers (dark markers vs prosocial markers per organism)